In [1]:
import sqlite3
con = sqlite3.connect("employee.db1")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              Phone_number TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (99, 'vikas' ,'1234567890' ,'sam Altman')
               """)

cursor.execute(""" INSERT INTO employee VALUES (100, 'mike' ,'0987654321' ,'Satya Nadella')
               """)

con.commit()
con.close()
print("database ready")

database ready


In [2]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db1")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager, Phone_number
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]},"
            f"Manager:{result[1]}",
            f"Phone Number:{result[2]}"
        )
    return "Employee not found"

In [3]:
print(db_query(99))

('Employee:vikas,Manager:sam Altman', 'Phone Number:1234567890')


In [4]:
from ddgs import DDGS
def web_search(query):
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    print(result)
    
    if len(result)==0:
        return "NO result found"
    return result[0]["body"]
    # print(result)

In [5]:
import time
def retry(func):
    def grace(*args):
        retry=3
        
        for attempt in range(retry):
            try:
                result=func(*args)
                return result
            except Exception as e:
                print(f"Retry {attempt +1}")
                time.sleep(1)
        
        return "Tool Failed"
    return grace

In [6]:
db_query_retry = retry(db_query)
web_search_retry=retry(web_search)

In [7]:
from langchain.tools import tool

/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
@tool
def db_query_tool(employee_id: int)-> str:
    """ retrieve Employee information from database"""
    return db_query_retry(employee_id)

In [9]:
@tool
def web_search_tool(query:str)->str:
    """ search in the internet for inforamtion"""
    return web_search_retry(query)

In [10]:
from langchain_core.tools import tool
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# MCP Server Configuration
server_params = StdioServerParameters(
    command="uvx",
    args=["mcp-server-calculator"]
)


@tool
async def calculator_tool(expression: str) -> str:
    """
    Use this tool whenever a mathematical calculation is required.

    Supports:
    - Addition
    - Subtraction
    - Multiplication
    - Division
    - Modulus
    - Exponents
    - Parentheses
    - Decimal arithmetic

    Input:
        A mathematical expression as a string.

    Example:
        "(25 + 35) / 5"

    Returns:
        The calculated result as a string.
    """

    try:
        async with stdio_client(server_params) as (read, write):

            async with ClientSession(read, write) as session:

                await session.initialize()

                result = await session.call_tool(
                    "calculate",
                    {
                        "expression": expression
                    }
                )

                return str(result)

    except Exception as e:
        return f"Calculator Error: {str(e)}"

In [11]:
import boto3
import json
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")


In [12]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        db_query_tool,
        calculator_tool,
        web_search_tool
    ],
    system_prompt="""
You are a ReAct AI assistant.

Your job is to solve user queries by  selecting the appropriate tool.

Available Tools:

1. db_query_tool
   - Use to retrieve employee information from the employee database.
   - Examples:
     • Who is employee 101?
     • Who manages employee 100?

2. calculator_tool
   - Use whenever mathematical calculations are required.
   - Examples:
     • (25 + 30) * 5
     • Calculate 15% of 2500
     • Solve ((10+20)/5)

3. web_search_tool
   - Use for current events, public information, or anything not present in the database.
   - Examples:
     • Who is the CEO of OpenAI?
     • Latest AI news
     • What is LangGraph?

Instructions:
You are a ReAct agent.

Think step-by-step internally.

Do NOT reveal your reasoning, thoughts, planning, or chain of thought.

Only return the final answer to the user.
"""
)

## Validators 

In [41]:
ALLOWED_TOPICS = {
    "employee",
    "database",
    "sql",
    "company",
    "calculate",
    "Add","Addition","Subtraction","Multiplication","Division",
    "math",
    "openai",
    "ai",
    "web search", "web",
    "technology"
}


def validate_topic(question: str):
    """
    Validate whether the user's query belongs to the supported topics.
    Raises a ValueError if the query is outside the allowed domain.
    """

    question = question.lower()

    if not any(topic in question for topic in ALLOWED_TOPICS):
        raise ValueError(
            "Rejected: This query is outside the supported topics."
        )

In [19]:
import re

def scrub_input_pii(question: str) -> str:
    """
    Mask common PII in the user input.
    """

    # Email
    question = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL]',
        question
    )

    # Phone Number
    question = re.sub(
        r'\b\d{10}\b',
        '[PHONE]',
        question
    )

    # Aadhaar
    question = re.sub(
        r'\b\d{4}\s?\d{4}\s?\d{4}\b',
        '[AADHAAR]',
        question
    )

    # PAN
    question = re.sub(
        r'\b[A-Z]{5}[0-9]{4}[A-Z]\b',
        '[PAN]',
        question
    )

    return question

In [24]:
import re

def scrub_output_pii(answer: str) -> str:
    """
    Mask common PII in the model output.
    """

    answer = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL]',
        answer
    )

    answer = re.sub(
        r'\b\d{10}\b',
        '[PHONE]',
        answer
    )

    answer = re.sub(
        r'\b\d{4}\s?\d{4}\s?\d{4}\b',
        '[AADHAAR]',
        answer
    )

    answer = re.sub(
        r'\b[A-Z]{5}[0-9]{4}[A-Z]\b',
        '[PAN]',
        answer
    )

    return answer

In [27]:
from pydantic import BaseModel

class AgentResponse(BaseModel):
    answer: str
    status: str


def validate_json(answer: str) -> AgentResponse:
    """
    Validate the final response against the expected schema.
    """

    return AgentResponse(
        answer=answer,
        status="Success"
    )

In [44]:
async def Agent(question):

    try:

        # Input Topic Validation
        validate_topic(question)

        # Input PII Scrubber
        question = scrub_input_pii(question)

        # ReAct Agent
        response = await agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": question
                    }
                ]
            }
        )

        answer = response["messages"][-1].content

        # Output PII Scrubber
        answer = scrub_output_pii(answer)

        # JSON Schema Validation
        final_response = validate_json(answer)

        return final_response.model_dump()

    except ValueError as e:
        return {
            "answer": str(e),
            "status": "Rejected"
        }

In [46]:
print(await Agent("who is the employee 100 and details ?"))

ServiceUnavailableException: An error occurred (ServiceUnavailableException) when calling the Converse operation (reached max retries: 4): Service capacity limit has been reached. Please try again later.

In [ ]:
print(await Agent("from web search who is prem Boinpally?"))

TypeError: 'CompiledStateGraph' object is not callable

In [39]:
print(await Agent("calculate 24+26"))

{'answer': 'The result of the calculation 24 + 26 is 50.', 'status': 'Success'}
